# 08 - Causal Production Decisioning

This notebook is your release gate for experiment-based product decisions.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from datetime import UTC, datetime
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

ab_json = ROOT / 'ml' / 'data' / 'reports' / 'causal' / 'ab_test' / 'ab_test_results.json'
did_json = ROOT / 'ml' / 'data' / 'reports' / 'causal' / 'did' / 'did_results.json'

assert ab_json.exists(), 'Run notebook 04 first (A/B).'
assert did_json.exists(), 'Run notebook 04 first (DiD).'

ab = json.loads(ab_json.read_text(encoding='utf-8'))
did = json.loads(did_json.read_text(encoding='utf-8'))


In [ ]:
# Cell 1: Decision thresholds
alpha = 0.05
min_binary_lift = 0.0
min_continuous_lift = 0.0

print('alpha:', alpha)
print('min_binary_lift:', min_binary_lift)
print('min_continuous_lift:', min_continuous_lift)


In [ ]:
# Cell 2: Evaluate A/B decision
binary = ab.get('binary_results', {})
cont = ab.get('continuous_results', {})
cuped = ab.get('cuped_results', {})

binary_pass = (binary.get('p_value', 1.0) < alpha) and (binary.get('rate_diff', -999) >= min_binary_lift)
continuous_pass = (cont.get('welch_p_value', 1.0) < alpha) and (cont.get('difference', -999) >= min_continuous_lift)
cuped_pass = (cuped.get('welch_p_value', 1.0) < alpha) and (cuped.get('difference', -999) >= min_continuous_lift)

ab_ship = bool(binary_pass or continuous_pass or cuped_pass)
print('A/B recommend ship:', ab_ship)


In [ ]:
# Cell 3: Evaluate DiD decision
reg = did.get('regression', {})
pt = did.get('parallel_trends_test', {})

coef = reg.get('coef_treated_post', 0.0)
pval = reg.get('p_value', 1.0)
pt_p = pt.get('p_value', None)
parallel_ok = True if pt_p is None else pt_p >= alpha

did_ship = bool((pval < alpha) and (coef >= min_continuous_lift) and parallel_ok)
print('DiD recommend ship:', did_ship)


In [ ]:
# Cell 4: Final causal release decision and report
final_decision = 'ship' if (ab_ship or did_ship) else 'iterate'

payload = {
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'ab_recommend_ship': ab_ship,
    'did_recommend_ship': did_ship,
    'final_decision': final_decision,
    'details': {
        'ab_binary_pass': binary_pass,
        'ab_continuous_pass': continuous_pass,
        'ab_cuped_pass': cuped_pass,
        'did_coef': coef,
        'did_p_value': pval,
        'did_parallel_trends_ok': parallel_ok,
    },
}

out_json = ROOT / 'ml' / 'data' / 'reports' / 'causal' / 'causal_release_decision.json'
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')

out_md = ROOT / 'ml' / 'data' / 'reports' / 'causal' / 'causal_release_decision.md'
out_md.write_text(
    f"""# Causal Release Decision

- final_decision: {final_decision}
- ab_recommend_ship: {ab_ship}
- did_recommend_ship: {did_ship}
""",
    encoding='utf-8',
)

print('Saved:', out_json)
print('Saved:', out_md)
print('Final decision:', final_decision)
